<a href="https://colab.research.google.com/github/sandroormeno/TUTORIAL-TENSORFLOW-LITE-ARDUINO-DUE/blob/master/monitor_exercice_part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%tensorflow_version 2.x

In [ ]:
import pandas
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [ ]:
print(tf.__version__)

In [ ]:
# Los datos
url  = 'https://raw.githubusercontent.com/sandroormeno/Neural-network-con-arduino-ATmega644/master/jupyter%20notebook/datos_aceler%C3%B3metro.csv'

In [ ]:
datos = pandas.read_csv(url)
datos[0:5]

In [ ]:
# Las 15 primeras columnas son los caracter?sticas
caracteristicas= datos.values[:,:15]
# La ?ltima columna se coonvierte en los etiquetas
etiquetas= datos.values[:,15]

In [ ]:
# Normalizamos las caraster?sticas. Las dividimos entre 255
caracteristicas_norm= caracteristicas/255

In [ ]:
# Codificaci?n en One hot
etiquetas__hot = to_categorical(etiquetas)

In [ ]:
# Dividimos los datos en estrenamiento y evaluaci?n
X_train, X_test, y_train, y_test = train_test_split(caracteristicas_norm, etiquetas__hot, test_size=0.2, random_state=21)

In [ ]:
# El modelo
model= tf.keras.Sequential()

model.add(layers.Dense(16, activation='relu', input_dim=15))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(3, activation='sigmoid'))

# Configulaci?n del entrenamiento
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# El entrenamiento
model.fit(X_train, y_train, epochs=300, batch_size=5)

In [ ]:
# La evaluaci?n
score = model.evaluate(X_test, y_test)
print("\n%s: %.2f%%" % (model.metrics_names[1], score[1]*100))

In [ ]:
# aqu? termina el 1er tutorial

In [ ]:
# Stay here
# Convert the model to the TensorFlow Lite format without quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
tflite_model = converter.convert()

# Save the model to disk
open("exercice_model.tflite", "wb").write(tflite_model)

In [ ]:
X_test = X_test.astype('float32')

# Instantiate an interpreter for each model
exercice_model = tf.lite.Interpreter('exercice_model.tflite')

# Allocate memory for each model
exercice_model.allocate_tensors()

# Get the input and output tensors so we can feed in values and get the results
input_details = exercice_model.get_input_details()
output_details = exercice_model.get_output_details()

test = 30

input_data = X_test[test].reshape((1,15))
exercice_model.set_tensor(input_details[0]['index'],input_data )

exercice_model.invoke()
  
output_data = exercice_model.get_tensor(output_details[0]['index'])
    
  
print(output_data)  
print(y_test[test])    

In [ ]:
# aqu? termina el 2do tutorial

In [ ]:
import os
basic_model_size = os.path.getsize("exercice_model.tflite")
print("Basic model is %d bytes" % basic_model_size)

In [ ]:
# Install xxd if it is not available
!apt-get -qq install xxd
# Save the file as a C source file
!xxd -i exercice_model.tflite > exercice_model.cc
# Print the source file
!cat exercice_model.cc